# 🚨 CrisisWorld — GRPO RL Training

**Model**: Qwen/Qwen2.5-0.5B-Instruct (default) or Meta-Llama-3-1B-Instruct  
**Algorithm**: GRPO (Group Relative Policy Optimization)  
**LoRA**: r=8, q_proj + v_proj  
**Exploration**: temperature=0.7, top_p=0.9, do_sample=True  

### Success Criteria
After training you should see in logs:
- `reward_std > 0`
- `loss > 0`
- `grad_norm > 0`
- rewards improving: `-80 → -40 → 10 → 60`

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU detected — switch runtime to GPU')

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
!git clone https://github.com/anilchowdary07/meta_x_scalar_ai_agents_city.git
%cd meta_x_scalar_ai_agents_city/backend

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q -r requirements-train.txt

In [ ]:
# ── 3. Set HuggingFace token (from Colab Secrets or paste here) ───────────────
import os

# Option A: Colab Secrets (recommended — never paste tokens in notebooks)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab Secrets')
except Exception:
    # Option B: Paste directly (remove before sharing)
    os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
    print('HF token set manually')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

In [ ]:
# ── 4. (Optional) Use smaller model if memory is tight ───────────────────────
# Default: Qwen/Qwen2.5-0.5B-Instruct (~1 GB)
# Uncomment for 1B Llama (needs ~4 GB VRAM):
# os.environ['CRISIS_MODEL'] = 'meta-llama/Meta-Llama-3-1B-Instruct'
print(f"Model: {os.getenv('CRISIS_MODEL', 'Qwen/Qwen2.5-0.5B-Instruct')}")

In [ ]:
# ── 5. Run training ───────────────────────────────────────────────────────────
# Expected output:
#   [reward batch] n=4  mean=-45.2  std=18.7  min=-80.1  max=-12.3
#   [GRPO step 5] reward=-45.2  reward_std=18.7  loss=0.043  grad_norm=0.82
#   [reward batch] n=4  mean=-28.1  std=22.4  ...
#   ...
#   rewards improving over time ✓

from train_grpo import train
train(
    samples=128,        # reduce for faster Colab run
    max_steps=100,
    horizon=10,         # steps per episode (reduce to speed up reward eval)
    logging_steps=5,
)

In [ ]:
# ── 6. Plot reward curve ──────────────────────────────────────────────────────
import json, matplotlib.pyplot as plt

with open('checkpoints/crisisworld-grpo/reward_curve.json') as f:
    data = json.load(f)

plt.figure(figsize=(10, 4))
plt.plot(data['episodes'], data['rewards'], color='#00ff88', linewidth=2)
plt.axhline(0, color='white', linestyle='--', alpha=0.3)
plt.title('CrisisWorld GRPO — Reward Curve', color='white', pad=12)
plt.xlabel('Episode', color='white')
plt.ylabel('Cumulative Reward', color='white')
plt.gca().set_facecolor('#060d1a')
plt.gcf().set_facecolor('#060d1a')
plt.tick_params(colors='white')
plt.tight_layout()
plt.savefig('checkpoints/crisisworld-grpo/reward_curve.png', dpi=120)
plt.show()
print('Reward curve saved.')

In [ ]:
# ── 7. Upload weights to HuggingFace Hub ─────────────────────────────────────
# Replace with your HF username/repo
HF_REPO = 'anilchowdary07/crisisworld-grpo-qwen'

from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('checkpoints/crisisworld-grpo')
tokenizer = AutoTokenizer.from_pretrained('checkpoints/crisisworld-grpo')
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f'Model pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 8. (Optional) Push artifacts back to GitHub ───────────────────────────────
# !git config user.email 'you@example.com'
# !git config user.name 'Your Name'
# !git add checkpoints/
# !git commit -m 'feat: add trained GRPO weights and reward curve'
# !git push
print('Uncomment the lines above to push artifacts to GitHub.')